In [21]:
import eazy
import eazy.photoz

In [22]:
import eazy

# 1. Load the filter master file
filter_file = eazy.filters.FilterFile()

# 2. Search for your target keyword (e.g., 'sdss' or 'ps1')
keyword = 'sdss'
print(f"--- Searching for filters matching '{keyword}' ---\n")

# Use enumerate(..., start=1) to keep track of the EAZY Filter ID
for EAZY_ID, entry in enumerate(filter_file.filters, start=1):
    if keyword in entry.name.lower():
        print(f"Filter ID: {EAZY_ID:<5} | Name: {entry.name}")

--- Searching for filters matching 'sdss' ---

Filter ID: 73    | Name: COSMOS/SDSS_filter_u.txt lambda_c= 3.5565e+03 AB-Vega= 0.938 w95=695.2
Filter ID: 74    | Name: COSMOS/SDSS_filter_g.txt lambda_c= 4.7025e+03 AB-Vega=-0.104 w95=1330.5
Filter ID: 75    | Name: COSMOS/SDSS_filter_r.txt lambda_c= 6.1766e+03 AB-Vega= 0.140 w95=1177.4
Filter ID: 76    | Name: COSMOS/SDSS_filter_i.txt lambda_c= 7.4961e+03 AB-Vega= 0.353 w95=1297.9
Filter ID: 77    | Name: COSMOS/SDSS_filter_z.txt lambda_c= 8.9467e+03 AB-Vega= 0.513 w95=1960.3
Filter ID: 156   | Name: SDSS/u.dat DR7+atm lambda_c= 3.5565e+03 AB-Vega= 0.938 w95=695.2
Filter ID: 157   | Name: SDSS/g.dat DR7+atm lambda_c= 4.7025e+03 AB-Vega=-0.104 w95=1330.5
Filter ID: 158   | Name: SDSS/r.dat DR7+atm lambda_c= 6.1756e+03 AB-Vega= 0.140 w95=1177.4
Filter ID: 159   | Name: SDSS/i.dat DR7+atm lambda_c= 7.4900e+03 AB-Vega= 0.352 w95=1297.0
Filter ID: 160   | Name: SDSS/z.dat DR7+atm lambda_c= 8.9467e+03 AB-Vega= 0.513 w95=1960.3
Filter ID: 278 

In [23]:
import eazy

# 1. Load the filter master file
filter_file = eazy.filters.FilterFile()

# 2. Search for your target keyword (e.g., 'sdss' or 'ps1')
keyword = 'ps1'
print(f"--- Searching for filters matching '{keyword}' ---\n")

# Use enumerate(..., start=1) to keep track of the EAZY Filter ID
for EAZY_ID, entry in enumerate(filter_file.filters, start=1):
    if keyword in entry.name.lower():
        print(f"Filter ID: {EAZY_ID:<5} | Name: {entry.name}")

--- Searching for filters matching 'ps1' ---

Filter ID: 334   | Name: PAN-STARRS/PS1.g lambda_c= 4.84911e+03 AB-Vega= -0.087
Filter ID: 335   | Name: PAN-STARRS/PS1.r lambda_c= 6.20119e+03 AB-Vega= 0.144
Filter ID: 336   | Name: PAN-STARRS/PS1.i lambda_c= 7.53496e+03 AB-Vega= 0.366
Filter ID: 337   | Name: PAN-STARRS/PS1.z lambda_c= 8.67418e+03 AB-Vega= 0.513
Filter ID: 338   | Name: PAN-STARRS/PS1.y lambda_c= 9.62777e+03 AB-Vega= 0.544


In [24]:
sdss_translate_content = """
seq id
petroMag_g  F157
petroMag_r  F158
petroMag_i  F159
petroMag_z  F160
"""
with open("sdss.translate", "w") as f:
    f.write(sdss_translate_content.strip())

In [25]:
ps1_translate_content = """
seq id
gMeanPSFMag F334
gMeanPSFMagErr  E334
rMeanPSFMag F335
rMeanPSFMagErr  E335
iMeanPSFMag F336
iMeanPSFMagErr  E336
zMeanPSFMag F337
zMeanPSFMagErr  E337
yMeanPSFMag F338
yMeanPSFMagErr  E338
"""
with open("ps1.translate", "w") as f:
    f.write(ps1_translate_content.strip())

In [26]:
import numpy as np
import pandas as pd

def convert_mag_to_flux_ujy(mag, mag_err, extinction=0.0):
    """
    Corrects for line-of-sight Galactic extinction and converts 
    AB magnitude to microjanskys (uJy).
    """
    # 1. Correct the raw magnitude for Milky Way dust extinction
    mag_corrected = mag - extinction
    
    # 2. Convert AB magnitude to microjanskys
    flux = 10**(-0.4 * (mag_corrected - 23.9))
    
    # 3. Propagate the symmetric magnitude error to linear flux error
    flux_err = flux * (0.4 * np.log(10) * mag_err)
    
    return flux, flux_err

# Example Workflow for a CSV target list:
# df = pd.read_csv("your_raw_dr19_data.csv")
# 
# If columns are already dust-extinction-corrected (e.g., SDSS dered_r):
# df['flux_r'], df['flux_err_r'] = convert_mag_to_flux_ujy(df['dered_r'], df['extinction_corrected_err_r'])
#
# df.to_csv("eazy_ready_input_catalog.cat", sep=' ', index=False)

In [27]:
# 1. 讀取星表
import pandas as pd
df = pd.read_csv('../data/merged_redshifts.csv')

# 2. 呼叫您既有的函數（將 mag_err 設為常數 0.1）
for band in ['g', 'r', 'i', 'z']:
    flux_col = f'flux_{band}'
    err_col = f'flux_err_{band}'
    
    # 呼叫您既有的 convert_mag_to_flux_ujy
    df[flux_col], df[err_col] = convert_mag_to_flux_ujy(df[f'petroMag_{band}'], df[f'petroMagErr_{band}'])
    
    # 過濾 -9999.0 無效值
    invalid = df[f'petroMag_{band}'] < 0
    df.loc[invalid, flux_col] = -99.0
    df.loc[invalid, err_col] = -99.0

# 3. 挑選並儲存為 cat 檔案
cat_df = df[['objid', 'ra', 'dec', 
             'flux_g', 'flux_err_g', 
             'flux_r', 'flux_err_r', 
             'flux_i', 'flux_err_i', 
             'flux_z', 'flux_err_z', 
             'redshift']].copy()
cat_df.rename(columns={'objid': 'id', 'redshift': 'z_spec'}, inplace=True)

with open('eazy_input.cat', 'w', encoding='utf-8') as f:
    f.write("# " + " ".join(cat_df.columns) + "\n")
    cat_df.to_csv(f, sep=' ', index=False, header=False)

# 4. 寫出 translate 對照檔
with open('eazy.translate', 'w', encoding='utf-8') as f:
    f.write("id id\nflux_g F157\nflux_err_g E157\nflux_r F158\nflux_err_r E158\nflux_i F159\nflux_err_i E159\nflux_z F160\nflux_err_z E160\nz_spec z_spec\n")

# 5. 將 zphot.param 中的 CATALOG_FILE 改為指向 eazy_input.cat
with open('zphot.param', 'r', encoding='utf-8') as f:
    lines = f.readlines()
with open('zphot.param', 'w', encoding='utf-8') as f:
    for line in lines:
        if line.startswith('CATALOG_FILE'):
            f.write('CATALOG_FILE         eazy_input.cat # Catalog data file\n')
        else:
            f.write(line)

print("完成！已使用您原有的轉換函數處理好 eazy_input.cat。")


完成！已使用您原有的轉換函數處理好 eazy_input.cat。


In [29]:
import os
import shutil
import subprocess
import eazy

# =========================================================================
# 1. 設置預設的 templates 與 filter 路徑 (Windows 安全清理版)
# =========================================================================

# 安全清理現有的 templates 連結或目錄
if os.path.lexists('./templates'):
    try:
        # 如果是 Junction 或是 Symlink，os.rmdir 是 Windows 上最安全的刪除方式（只切斷連結，不刪除目標內容）
        os.rmdir('./templates')
        print("Removed existing templates link")
    except Exception:
        try:
            if os.path.isdir('./templates') and not os.path.islink('./templates'):
                shutil.rmtree('./templates')
                print("Removed existing templates directory")
            else:
                os.unlink('./templates')
                print("Unlinked existing templates")
        except Exception as e:
            print(f"Warning: Could not clear templates path: {e}")

# 獲取 eazy 內建的 templates 與 filters 路徑
eazy_data_path = getattr(eazy, 'DATA_PATH', None)
if eazy_data_path is None:
    eazy_data_path = os.path.join(os.path.dirname(eazy.__file__), 'data')

t_path = os.path.abspath(os.path.join(eazy_data_path, 'templates'))
res_path = os.path.abspath(os.path.join(eazy_data_path, 'filters/FILTER.RES.latest'))

# 1-1. 建立 templates (目錄)
try:
    if os.name == 'nt':
        # 在 Windows 上使用 Junction 目錄聯接，不需要管理員特權
        subprocess.run(f'mklink /J "{os.path.abspath("./templates")}" "{t_path}"', shell=True, check=True)
        print("Created directory junction for templates")
    else:
        os.symlink(t_path, './templates')
        print("Created symlink for templates")
except Exception as e:
    # 若連結失敗則退回複製模式
    print(f"Link failed ({e}), falling back to copy...")
    shutil.copytree(t_path, './templates')
    print("Copied templates directory")

# 1-2. 建立 FILTER.RES.latest (檔案)
if os.path.lexists('./FILTER.RES.latest'):
    try:
        os.unlink('./FILTER.RES.latest')
    except Exception:
        try: os.remove('./FILTER.RES.latest')
        except: pass

try:
    if os.name == 'nt':
        # 在 Windows 上使用 Hardlink 硬連結，不需要管理員特權
        subprocess.run(f'mklink /H "{os.path.abspath("./FILTER.RES.latest")}" "{res_path}"', shell=True, check=True)
        print("Created hardlink for FILTER.RES.latest")
    else:
        os.symlink(res_path, './FILTER.RES.latest')
        print("Created symlink for FILTER.RES.latest")
except Exception as e:
    # 若連結失敗則退回複製模式
    print(f"Link failed ({e}), falling back to copy...")
    shutil.copy2(res_path, './FILTER.RES.latest')
    print("Copied FILTER.RES.latest")

# =========================================================================
# 2. 編輯或生成本地的參數設定檔
# =========================================================================
param_file = 'zphot.param'

# =========================================================================
# 3. 選擇適當的 translation map
# =========================================================================
translate_file = 'eazy.translate' 

# =========================================================================
# 4. 初始化核心 PhotoZ 框架
# =========================================================================
ez = eazy.photoz.PhotoZ(param_file=param_file, 
                        translate_file=translate_file, 
                        zeropoint_file=None, 
                        load_templates=True, 
                        write_results=True)

# =========================================================================
# 5. 執行平行運算模板擬合優化
# =========================================================================
print("Starting photo-z template fitting calculations...")
ez.fit_parallel(n_proc=6)
print("Calculations complete. Output hdf5 and catalog files written to disk.")
# 將擬合結果正式輸出至硬碟（會自動建立 OUTPUT 目錄並生成 photz.zout.fits 等檔案）
ez.standard_output()


Removed existing templates link
Created directory junction for templates
Created hardlink for FILTER.RES.latest
Read default param file: zphot.param
Read CATALOG_FILE: eazy_input.cat
   >>> NOBJ = 923
flux_g flux_err_g (157): SDSS/g.dat
flux_r flux_err_r (158): SDSS/r.dat
flux_i flux_err_i (159): SDSS/i.dat
flux_z flux_err_z (160): SDSS/z.dat
Set sys_err = 0.03 (positive=True)
Read PRIOR_FILE:  templates/prior_F160W_TAO.dat
Template grid: templates/fsps_full/tweak_fsps_QSF_12_v3.param (this may take some time)


100%|██████████| 12/12 [00:11<00:00,  1.02it/s]


Template   0: tweak_fsps_QSF_12_v3_001.dat (NZ=1).
Template   1: tweak_fsps_QSF_12_v3_002.dat (NZ=1).
Template   2: tweak_fsps_QSF_12_v3_003.dat (NZ=1).
Template   3: tweak_fsps_QSF_12_v3_004.dat (NZ=1).
Template   4: tweak_fsps_QSF_12_v3_005.dat (NZ=1).
Template   5: tweak_fsps_QSF_12_v3_006.dat (NZ=1).
Template   6: tweak_fsps_QSF_12_v3_007.dat (NZ=1).
Template   7: tweak_fsps_QSF_12_v3_008.dat (NZ=1).
Template   8: tweak_fsps_QSF_12_v3_009.dat (NZ=1).
Template   9: tweak_fsps_QSF_12_v3_010.dat (NZ=1).
Template  10: tweak_fsps_QSF_12_v3_011.dat (NZ=1).
Template  11: tweak_fsps_QSF_12_v3_012.dat (NZ=1).
Process templates: 12.186 s


109it [00:00, 9625.63it/s]


Starting photo-z template fitting calculations...


...

Compute best fits
fit_best: 0.3 s (n_proc=1,  NOBJ=853)
Fit 5.6 s (n_proc=6, NOBJ=923)
Calculations complete. Output hdf5 and catalog files written to disk.
Get best fit coeffs & best redshifts
fit_best: 0.8 s (n_proc=1,  NOBJ=853)
Get parameters (UBVJ=[153, 154, 155, 161], simple=False)


100%|██████████| 853/853 [00:03<00:00, 241.19it/s]
c:\Users\nianz\micromamba\envs\astro13\Lib\site-packages\eazy\photoz.py:4283: RuntimeWarning: invalid value encountered in divide
  coeffs_norm = (coeffs_norm.T/coeffs_norm.sum(axis=1)).T
c:\Users\nianz\micromamba\envs\astro13\Lib\site-packages\eazy\photoz.py:4318: RuntimeWarning: invalid value encountered in divide
  draws_norm = (draws_norm.T/draws_norm.sum(axis=2).T).T
c:\Users\nianz\micromamba\envs\astro13\Lib\site-packages\astropy\units\quantity.py:676: RuntimeWarning: divide by zero encountered in divide
  result = super().__array_ufunc__(function, method, *arrays, **kwargs)
c:\Users\nianz\micromamba\envs\astro13\Lib\site-packages\astropy\units\quantity.py:676: RuntimeWarning: invalid value encountered in multiply
  result = super().__array_ufunc__(function, method, *arrays, **kwargs)
100%|██████████| 853/853 [00:07<00:00, 114.13it/s]


Abs Mag filters [271, 272, 274]
Rest-frame filters:
~~~~~~~~~~~~~~~~~~~ 
   0 RestUV/Tophat_1700_200.dat lambda_c= 1.6989e+03 AB-Vega=1.916 w95=190.7
   1 RestUV/Tophat_2200_200.dat lambda_c= 2.1993e+03 AB-Vega=1.691 w95=191.1
   2 RestUV/Tophat_2800_200.dat lambda_c= 2.7996e+03 AB-Vega=1.465 w95=191.2


100%|██████████| 853/853 [00:02<00:00, 410.82it/s]
c:\Users\nianz\micromamba\envs\astro13\Lib\site-packages\eazy\photoz.py:4122: RuntimeWarning: divide by zero encountered in log10
  obsm = self.param.params['PRIOR_ABZP'] - 2.5*np.log10(rf[:,i,:])
c:\Users\nianz\micromamba\envs\astro13\Lib\site-packages\eazy\photoz.py:4122: RuntimeWarning: invalid value encountered in log10
  obsm = self.param.params['PRIOR_ABZP'] - 2.5*np.log10(rf[:,i,:])


(<Table length=923>
          id         ...                  ABSM_274                
                     ...                                          
        int64        ...                 float64[5]               
 ------------------- ... -----------------------------------------
 1237663543696621695 ...                nan .. -20.541583100069936
 1237663543696621698 ...                nan .. -20.669878084767728
 1237663543696621713 ...                                nan .. nan
 1237653651839975466 ...                 nan .. -19.76827457596918
 1237653651839975718 ...                nan .. -18.182691213151468
 1237651497364750398 ...                 nan .. -19.33259446924719
 1237663530252370209 ...                nan .. -17.392748026577156
 1237663531327488471 ...                 nan .. -18.03870585889694
 1237663531327488441 ...                nan .. -21.460107925241243
                 ... ...                                       ...
 1237661082662666676 ...                na